In [ ]:
import rasterio
from rasterio.windows import Window
from rasterio.env import Env
import numpy as np
import psutil
import os

red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"
tile_size = 512

process = psutil.Process(os.getpid())
with Env(GDAL_CACHEMAX=128): 
    
    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:

        height = red_src.height
        width = red_src.width

        print(f"Image size: {width} x {height}")
        for row in range(0,height,tile_size):
            for col in range(0,width,tile_size):

                win_h=min(tile_size , height-row)
                win_w= min(tile_size , width - col)

                window = Window(col , row , win_w , win_h)
                red = red_src.read(1,window=window).astype("float32")/10000.0
                nir = nir_src.read(1,window= window).astype("float32")/10000.0
                if np.all(red==0) and np.all(nir==0):
                    continue
                denominator = nir + red
                denominator[denominator == 0] = 1e-6

                ndvi = (nir - red) / denominator 
                mean = np.mean(ndvi)
                print(f"The mean NDVI is {mean:4f}")

                mem = process.memory_info().rss / (1024**2)
                print(f"The memory usage is {mem:2f} MB")

                del red
                del nir
                del ndvi


In [ ]:
import rasterio
from rasterio.windows import Window 
from rasterio.env import Env
import time 
import psutil
import os

red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"
tile_size = 512
tile_count=0
max=0.0
latitude , longitude = 0,0
start_time_ = time.time()
with Env(GDAL_CACHEMAX=128):
    with rasterio.open(red_path) as red_src , rasterio.open(nir_path) as nir_src:
        height = red_src.height
        width = red_src.width
        for row in range(0,height,tile_size):
            for col in range(0,width,tile_size):
                min_h= min(tile_size,height-row)
                min_w=min(tile_size,width-col)

                window = Window(col,row,min_w,min_h)
                red = red_src.read(1,window = window).astype('float32')/10000.0
                nir= nir_src.read(1,window=window).astype('float32')/10000.0
                if np.all(red==0) and np.all(nir)==0:
                    continue 
                denom = nir + red
                denom[denom == 0] = 1e-6
                ndvi=(nir-red)/denom
                transform = red_src.window_transform(window)
                lon , lat = transform*(0,0)
                if mean_ndvi>max:
                    max = mean_ndvi
                    latitude,longitude = lat,lon
                mean_ndvi = np.mean(ndvi)
                mem = process.memory_info().rss /1024**2
                print(f"At lat : {lat}, lon {lon} the ndvi is : {mean_ndvi}")
                print(f"The total memory used is {mem}")
                tile_count +=1
                del red , nir , ndvi
            
elapsed_time = time.time() - start_time_

print("--------------------------------")
print(f"tiles processed = {tile_count}")
print(f"Total time processed = {elapsed_time}")
print(f"Tile processing speed = {tile_count/(elapsed_time)} per second")


At lat : 2000040.0, lon 699960.0 the ndvi is : 0.3622794449329376
The total memory used is 101.16015625
At lat : 2000040.0, lon 705080.0 the ndvi is : 0.3832753300666809
The total memory used is 99.01171875
At lat : 2000040.0, lon 710200.0 the ndvi is : 0.3623908460140228
The total memory used is 99.046875
At lat : 2000040.0, lon 715320.0 the ndvi is : 0.3424648344516754
The total memory used is 99.046875
At lat : 2000040.0, lon 720440.0 the ndvi is : 0.2841734290122986
The total memory used is 99.0546875
At lat : 2000040.0, lon 725560.0 the ndvi is : 0.2454030066728592
The total memory used is 99.05859375
At lat : 2000040.0, lon 730680.0 the ndvi is : 0.2595146596431732
The total memory used is 99.05859375
At lat : 2000040.0, lon 735800.0 the ndvi is : 0.2746123671531677
The total memory used is 99.06640625
At lat : 2000040.0, lon 740920.0 the ndvi is : 0.04608573019504547
The total memory used is 99.1015625
At lat : 1994920.0, lon 699960.0 the ndvi is : 0.38153478503227234
The total 